## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para disponer de persistencia de archivos

In [126]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Drive already mounted at /content/.drive; to attempt to forcibly remount, call drive.mount("/content/.drive", force_remount=True).


Bajar datasets si hace falta

In [127]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Modelo Regresion Lineal

## 1.1 Init Experimento

In [128]:
# instalacion de paquetes que NO vienen por default en Colab
!pip install uv
!uv pip install -q kaggle
!uv pip install -q statsmodels

In [129]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)


In [130]:
import os as os
import numpy as np
import polars as pl
import polars.selectors as cs
import statsmodels.api as sm

import warnings
warnings.filterwarnings("ignore")

Por favor, cargar aqui SU semilla primigenia

In [131]:
# defino los parametros
PARAM = {'experimento':'LR02',
  'kaggle_competition':'labo-iii-2026-rosario',
  'semilla_primigenia':19970220,
  'empiojar_ruido':0.0
}

In [132]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/exp/LR02


## 1.2 Preprocesamiento

In [153]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [154]:
dataset.head(10)

periodo,customer_id,product_id,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn
i64,i64,i64,i64,i64,f64,f64
201701,10234,20524,0,2,0.053,0.053
201701,10032,20524,0,1,0.13628,0.13628
201701,10217,20524,0,1,0.03028,0.03028
201701,10125,20524,0,1,0.02271,0.02271
201701,10012,20524,0,11,1.54452,1.54452
201701,10080,20524,0,1,0.01514,0.01514
201701,10015,20524,0,4,0.106,0.106
201701,10062,20524,0,1,0.18928,0.18928
201701,10159,20524,0,3,0.02271,0.02271


In [134]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [135]:
# Quito el mes de las PASO
tb_ventas = tb_ventas.filter(
    pl.col("periodo") != 201908
)

In [136]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

30314
21578


In [137]:
display( tb_ventas )

product_id,periodo,tn
i64,i64,f64
20001,201701,934.77222
20001,201702,798.0162
20001,201703,1303.35771
20001,201704,1069.9613
20001,201705,1502.20132
…,…,…
21276,201907,0.00223
21276,201909,0.01856
21276,201910,0.02079


### 1.2.1 Empiojado  lognormal

In [138]:

if PARAM['empiojar_ruido']>0.0:
  np.random.seed(PARAM['semilla_primigenia'])
  tb_ventas = tb_ventas.sort(["product_id", "periodo"])
  # vector con el ruido multiplicativo de media 1.0  y desvio  'empiojar_ruido'
  noise_multiplier = np.random.lognormal(mean=0.0, sigma=PARAM['empiojar_ruido'], size=tb_ventas.height)

  tb_ventas = tb_ventas.with_columns(
    (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
  )


### 1.2.2  Dataset aplanado con lags

In [139]:
lags = [-2, *range(0,12)]

tb_lags = (
    tb_ventas.sort(["product_id", "periodo"])
      .with_columns(
          [
              pl.col("tn")
                .shift(lag)
                .over("product_id")
                .alias(f"tn_{lag}")
              for lag in lags
          ]
      )
)

tb_lags = tb_lags.rename({"tn_-2": "clase"})

In [140]:
display(tb_lags)

product_id,periodo,tn,clase,tn_0,tn_1,tn_2,tn_3,tn_4,tn_5,tn_6,tn_7,tn_8,tn_9,tn_10,tn_11
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
20001,201701,934.77222,1303.35771,934.77222,null,null,null,null,null,null,null,null,null,null,null
20001,201702,798.0162,1069.9613,798.0162,934.77222,null,null,null,null,null,null,null,null,null,null
20001,201703,1303.35771,1502.20132,1303.35771,798.0162,934.77222,null,null,null,null,null,null,null,null,null
20001,201704,1069.9613,1520.06539,1069.9613,1303.35771,798.0162,934.77222,null,null,null,null,null,null,null,null
20001,201705,1502.20132,1030.67391,1502.20132,1069.9613,1303.35771,798.0162,934.77222,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
21276,201907,0.00223,0.02079,0.00223,0.04086,0.09283,0.10173,0.12249,null,null,null,null,null,null,null
21276,201909,0.01856,0.03341,0.01856,0.00223,0.04086,0.09283,0.10173,0.12249,null,null,null,null,null,null
21276,201910,0.02079,0.00892,0.02079,0.01856,0.00223,0.04086,0.09283,0.10173,0.12249,null,null,null,null,null


### 1.2.4  Definicion de Training

In [141]:
# Esta es la salsa mágica del notebook

productos_magicos = [ 20002, 20003, 20005, 20006, 20009, 20010, 20011, 20013, 20015,
  20017, 20018, 20019, 20021, 20022, 20026, 20028, 20035, 20038, 20042, 20043, 20044,
  20045, 20046, 20049, 20051, 20052, 20055, 20081, 20324, 20417, 20771, 20947, 20949, 21003
]

productos_magicos = [ 20001, 20002, 20005, 20013, 20033, 20037, 20038, 20043, 20044,
  20045, 20046, 20052, 20055, 20058, 20059, 20069, 20070, 20072, 20073, 20075, 20080,
  20091, 20094, 20099, 20107, 20114, 20120, 20132, 20137, 20139, 20142, 20144, 20146,
  20148, 20151, 20153, 20157, 20158, 20161, 20162, 20166, 20167, 20189, 20198, 20201,
  20202, 20203, 20208, 20226, 20228, 20231, 20233, 20253, 20254, 20256, 20269, 20270,
  20271, 20275, 20276, 20277, 20278, 20288, 20298, 20315, 20317, 20320, 20322, 20335,
  20337, 20338, 20344, 20348, 20350, 20353, 20359, 20385, 20390, 20398, 20402, 20403,
  20406, 20411, 20416, 20417, 20418, 20419, 20421, 20422, 20424, 20428, 20429, 20443,
  20456, 20466, 20469, 20479, 20497, 20500, 20509, 20514, 20517, 20524, 20532, 20549,
  20551, 20560, 20561, 20565, 20568, 20579, 20583, 20585, 20586, 20589, 20599, 20606,
  20614, 20624, 20632, 20642, 20646, 20653, 20655, 20657, 20660, 20661, 20663, 20666,
  20677, 20680, 20684, 20696, 20699, 20713, 20737, 20744, 20745, 20765, 20768, 20773,
  20777, 20786, 20789, 20800, 20807, 20812, 20818, 20830, 20832, 20838, 20847, 20855,
  20863, 20864, 20882, 20883, 20906, 20913, 20914, 20919, 20922, 20925, 20937, 20945,
  20956, 20961, 20965, 20970, 20976, 20986, 20996, 21016, 21038, 21048, 21049, 21077,
  21080, 21088, 21118, 21170, 21200
]

In [142]:
productos_no_magicos = [
    p for p in tb_ventas["product_id"].unique().to_list()
    if p not in productos_magicos
]

In [143]:
import pandas as pd

tb_ventas_pd["fecha"] = pd.to_datetime(
    tb_ventas_pd["periodo"].astype(str) + "01",
    format="%Y%m%d"
)

magicos = (
    tb_ventas_pd[tb_ventas_pd["product_id"].isin(productos_magicos)]
    .groupby("fecha")["tn"]
    .sum()
)

no_magicos = (
    tb_ventas_pd[tb_ventas_pd["product_id"].isin(productos_no_magicos)]
    .groupby("fecha")["tn"]
    .sum()
)

total = (
    tb_ventas_pd
    .groupby("fecha")["tn"]
    .sum()
)

In [144]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=magicos.index,
        y=magicos.values,
        mode="lines",
        name="Mágicos"
    )
)

fig.add_trace(
    go.Scatter(
        x=no_magicos.index,
        y=no_magicos.values,
        mode="lines",
        name="No mágicos"
    )
)

fig.add_trace(
    go.Scatter(
        x=total.index,
        y=total.values,
        mode="lines",
        name="Total"
    )
)

fig.update_layout(
    title="Evolución del sell-in",
    xaxis_title="Fecha",
    yaxis_title="tn",
    width=1200,
    height=600,
    hovermode="x unified",
    legend=dict(
        x=1.02,
        y=0.5,
        xanchor="left",
        yanchor="middle"
    )
)

fig.show()

In [145]:
# Entreno con los datos de 2018 para los  productos_magicos

dtrain = tb_lags.filter( (pl.col("periodo") == 201812) & (pl.col("product_id").is_in(productos_magicos)) )

display( dtrain )

product_id,periodo,tn,clase,tn_0,tn_1,tn_2,tn_3,tn_4,tn_5,tn_6,tn_7,tn_8,tn_9,tn_10,tn_11
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
20001,201812,1486.68669,1259.09363,1486.68669,1813.01511,2295.19832,1438.67455,1800.96168,1470.41009,1150.79169,1293.89788,1251.28462,1856.83534,1043.7647,1169.07532
20002,201812,1009.45458,1043.01349,1009.45458,1766.81068,1378.49032,954.23575,1161.8843,977.40239,1033.82845,1103.39191,999.20934,966.86044,712.00087,984.80167
20005,201812,372.63428,409.8995,372.63428,469.26344,893.74086,761.7752,874.88924,502.34077,547.62513,637.11135,496.41774,559.98671,399.20878,417.53208
20013,201812,333.70155,377.10855,333.70155,367.82928,469.93401,235.49526,437.85378,391.482,432.0225,494.79885,448.49259,593.35731,382.2273,329.57379
20033,201812,150.48852,132.55515,150.48852,253.0437,278.24706,300.47745,266.28693,207.81852,175.15407,246.47805,190.81062,228.00414,163.39323,171.61053
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
21080,201812,0.41052,0.1834,0.41052,0.39856,0.30683,0.42696,0.5951,0.6628,0.62566,0.50007,0.76106,0.54048,0.27629,0.42806
21088,201812,0.22682,0.49756,0.22682,0.4234,0.33713,0.55409,0.51011,0.50419,0.81655,0.80695,0.64965,0.91654,0.49415,0.37283
21118,201812,0.31089,0.4323,0.31089,0.27071,0.32891,0.38184,0.46436,0.48293,0.46226,0.67321,0.48212,0.75515,0.52254,0.76033


## 1.3  Modelo de Regresion Lineal

In [146]:

# campos a utilizar
campos_buenos = dtrain.select(cs.starts_with("tn_"))


# tristemente paso por pandas
X_train = dtrain.select(campos_buenos).to_pandas()

# artificio para agregar intercepto
X_train = sm.add_constant(X_train)

# tristemente paso por pandas
y_train = dtrain['clase'].to_pandas()


modelo = sm.OLS(y_train, X_train).fit()
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                  clase   R-squared:                       0.990
Model:                            OLS   Adj. R-squared:                  0.989
Method:                 Least Squares   F-statistic:                     1336.
Date:                Fri, 05 Jun 2026   Prob (F-statistic):          1.41e-160
Time:                        19:17:59   Log-Likelihood:                -730.19
No. Observations:                 182   AIC:                             1486.
Df Residuals:                     169   BIC:                             1528.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0706      1.259      0.056      0.9

## 1.4 Aplicacion a los datos del futuro

### 1.4.1  Definicion datos del futuro
Solo puedo aplicar al modelo a los datos que tengan completos todos los meses
<br> de los 780 productos a predecir
* 656 tienen todos los meses del 2019 completos
* 124 no (los voy a predecir con el simple promedio)

In [147]:
dfuture = tb_lags.filter( (pl.col("periodo") == 201912) & (pl.col("tn_11").is_not_null() ))

display( dfuture )

product_id,periodo,tn,clase,tn_0,tn_1,tn_2,tn_3,tn_4,tn_5,tn_6,tn_7,tn_8,tn_9,tn_10,tn_11
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
20001,201912,1504.68856,null,1504.68856,1397.37231,1561.50552,1660.00561,1678.99318,1109.93769,1629.78233,1647.63848,1470.65653,1259.09363,1275.77351,1486.68669
20002,201912,1087.30855,null,1087.30855,1423.57739,1979.53635,1090.18771,1066.44999,928.36431,1034.98927,1287.62346,1083.62552,1043.01349,1266.78751,1009.45458
20003,201912,892.50129,null,892.50129,948.29393,1081.36645,967.77116,715.20314,662.38654,590.12515,565.33774,638.0401,758.32657,964.76919,769.82869
20004,201912,637.90002,null,637.90002,723.94206,1064.69633,786.1714,521.71519,667.19411,603.31081,466.70901,619.77084,441.70332,511.33713,585.56477
20005,201912,593.24443,null,593.24443,606.91173,996.78275,879.52808,745.74978,876.39696,897.26297,624.9988,488.21387,409.8995,363.58438,372.63428
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
21248,201912,0.01129,null,0.01129,0.02964,0.0127,0.01411,0.02116,0.00988,0.01553,0.03106,0.05365,0.06209,0.02962,0.09601
21256,201912,0.01271,null,0.01271,0.02682,0.00847,0.00423,0.02822,0.00988,0.01553,0.01835,0.0593,0.05081,0.03811,0.05788
21259,201912,0.01412,null,0.01412,0.02965,0.01975,0.00564,0.04657,0.00988,0.01976,0.02117,0.06777,0.0508,0.04234,0.05928


### 1.4.2  Aplicacion del modelo de regresion a los datos del futuro

Se aplica el modelo a los 656 datos del futuro

In [148]:

# campos a utilizar
campos_buenos = dfuture.select(cs.starts_with("tn_"))

# tristemente paso por pandas
X_future = dfuture.select(campos_buenos).to_pandas()

# artificio para agregar intercepto
X_future = sm.add_constant(X_future)

# la prediccion
prediccion = modelo.predict(X_future)


tb_regresion = dfuture.select(['product_id']).with_columns(
    pl.Series("tn_pred", prediccion)
)

display( tb_regresion)

product_id,tn_pred
i64,f64
20001,1287.22826
20002,1141.803974
20003,763.911698
20004,531.077663
20005,450.258598
…,…
21248,0.112298
21256,0.106183
21259,0.108129


### 1.4.3  Join de los modelos de regresion y promedio

In [149]:
# tabla conh los promedios
primer_periodo = 201901
ultimo_periodo = 201912
tb_meses12 = tb_ventas.filter( pl.col("periodo").is_between(primer_periodo,ultimo_periodo)).group_by("product_id").agg(
 pl.col("tn").mean().alias("tn"))

tb_meses12 = tb_meses12.select(["product_id", "tn"])
display( tb_meses12 )

product_id,tn
i64,f64
20001,1472.313395
20002,1208.314868
20003,798.556478
20004,640.404565
20005,680.233932
…,…
21263,0.02964
21265,0.09772
21266,0.103532


In [150]:
# Update table1 using table2
k = 1.00 # Sesgar

tb_final = (
    tb_meses12
    .join(tb_regresion, on="product_id", how="left")
    .with_columns(
        pl.coalesce([
            pl.col("tn_pred") * k,
            pl.col("tn")
        ]).alias("tn")
    )
    .drop("tn_pred")
)

display(tb_final)

product_id,tn
i64,f64
20001,1287.22826
20002,1141.803974
20003,763.911698
20004,531.077663
20005,450.258598
…,…
21263,0.112354
21265,0.09772
21266,0.103532


## 1.5 Submit a Kaggle

In [151]:
# Submit a Kaggle
if PARAM['empiojar_ruido']<=0.0:
  archivo= "linreg.csv"
  mensaje= "Regresion Lineal"
else:
  archivo= "linreg_empiojado.csv"
  mensaje= "Linear Regression logEMPIOJADO al " + str(PARAM['empiojar_ruido'])

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )

[![Ver video](https://www.youtube.com/embed/IY9G79BDRS4)](https://www.youtube.com/embed/IY9G79BDRS4)

## 1.6 Extra

In [181]:
import pandas as pd
import plotly.express as px

cliente = 10400
producto = 20001

serie = (
    dataset
    .filter(
        (pl.col("customer_id") == cliente) &
        (pl.col("product_id") == producto)
    )
    .group_by("periodo")
    .agg(pl.col("tn").sum().alias("ventas"))
    .sort("periodo")
)

serie_pd = serie.to_pandas()

serie_pd["fecha"] = pd.to_datetime(
    serie_pd["periodo"].astype(str) + "01",
    format="%Y%m%d"
)

fig = px.line(
    serie_pd,
    x="fecha",
    y="ventas",
    title=f"Ventas producto {producto} - Cliente {cliente}"
)

fig.update_layout(
    width=1200,
    height=600
)

fig.show()